In [30]:
import pandas as pd


In [31]:
componentes = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\componentes.parquet')
cursos = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\cursos.parquet')
discentes = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\discentes.parquet')
matriculas = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\matriculas.parquet')

# não existe relacionamento por id, solução: criar um agrupametno por em nivel em departamento
docentes = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\docentes.parquet')

In [32]:
# garantir unicidade das linhas
discentes.drop_duplicates(subset=['id_discente'], inplace=True, ignore_index=True)

df = (
    discentes.merge(matriculas, on ='id_discente', how='right')
    .merge(componentes, on='id_disciplina',how= 'left')
    .merge(cursos, on = ['id_estrutura_curricular','id_disciplina'], how='left', suffixes=['','_cursos'])
)

In [33]:
df1 = df.copy()

col = list(discentes.columns) + ['situacao', 'id_disciplina', 'id_curso_cursos','ano','periodo','ch_aula', 'ch_laboratorio', 'ch_total', 'cr_aula', 'cr_laboratorio',
       'cr_estagio', 'ch_ead','sigla_departamento','nome_componete_curricular','curso_nome']
x1 =(df1[col])

# funções auxiliares

def gerar_dummies(df,coluna, map=None):
    if map is not None:
        df[coluna] = df[coluna].map(map)
    
    else:
        df[coluna] = df[coluna].astype(str).str.lower()
        
    return pd.get_dummies(df, columns=[coluna],drop_first=True,dtype=int)

# dummies sexo 
x1.sexo.value_counts(dropna=False)

x1 = gerar_dummies(x1,'sexo',{'F':'feminino', 'M':'masculino'})

# estado civil

x1 = gerar_dummies(x1, 'estado_civil', {'Solreiro(a)':'solteiro', 'Casado(a)':'casado','Outro':'outro'})

# raça declarada

x1 = gerar_dummies(x1, 'raca_declarada')

# renda familiar
x1 = gerar_dummies(x1, 'faixa_renda_familiar')

# disciplina
# x1 = gerar_dummies(x1,'id_disciplina')

# curso
x1 = gerar_dummies(x1,'id_curso_cursos')

x1 = gerar_dummies(x1,'sigla_departamento')


C:\Users\Alef\AppData\Local\Temp\ipykernel_10020\1588113703.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[coluna] = df[coluna].map(map)


In [34]:
x1.situacao.value_counts()

x1 =x1.loc[x1['situacao'].isin(['APROVADO','APROV. C/ DISTINÇÃO','REPROVADO','REP. MEDIA E FALTA', 'REP. FALTA'])]
x1.situacao.value_counts()


x1 = gerar_dummies(x1, 'situacao',{'APROVADO':'aprovado','REPROVADO':'reprovado','REP. MEDIA E FALTA': 'rep_md_flt','REP. FALTA':'rep_flt'})

x1['reprovado'] = (
    x1[['situacao_reprovado',
        'situacao_rep_md_flt',
        'situacao_rep_flt']]
    .sum(axis=1)
    .clip(upper=1)
)

In [23]:
dms = pd.get_dummies(x1['id_disciplina'], drop_first=True, dtype='int8')
print(dms.shape)


print("Shape atual:", x1.shape)
print("N colunas:", len(x1.columns))
print("Disciplinas únicas:", x1['id_disciplina'].nunique())

(357428, 1542)
Shape atual: (357428, 67)
N colunas: 67
Disciplinas únicas: 1543


## Engenharia de Features

# criando variável tem laboratório e tem EAD

In [38]:
import numpy as np
x1['tem_laboratorio'] = np.where(
    x1['ch_laboratorio'].fillna(0) > 0,
    1,
    0
).astype(int)

# --------------------------------------------------
# 4) dummy: tem EAD?
# tem_ead = 1 se ch_ead > 0, senão 0
# --------------------------------------------------
x1['tem_ead'] = np.where(
    x1['ch_ead'].fillna(0) > 0,
    1,
    0
).astype(int)

## Criando variável de disciplina do ensino técnico

In [39]:
x1['dummy_tecnico'] = (
    ~x1['nome_componete_curricular'].str.contains('º|°', na=False)
).astype(int)

## Reprovações até então e reprovaçãoes até então na disciplina

In [40]:

# criando variável reprovações até então
x1['ano_periodo'] = (x1['ano']+'.'+x1['periodo']).astype(float)

reprov_ate_entao = (
    x1.groupby(['id_discente', 'ano_periodo'])['reprovado']
      .sum()
      .sort_index(level=['id_discente', 'ano_periodo'])
      .groupby(level=0)
      .cumsum()
      .groupby(level=0)
      .shift(1)
      .fillna(0)
      .reset_index(name='reprov_ate_entao')
)

x1 = x1.merge(
    reprov_ate_entao,
    on=['id_discente', 'ano_periodo'],
    how='left'
)
# criando variável reprovações na disciplina até então
reprov_disc_ate_entao = (
    x1.groupby(['id_discente', 'id_disciplina', 'ano_periodo'])['reprovado']
      .sum()
      .sort_index(level=['id_discente', 'id_disciplina', 'ano_periodo'])
      .groupby(level=[0, 1])
      .cumsum()
      .groupby(level=[0, 1])
      .shift(1)
      .fillna(0)
      .reset_index(name='reprov_disc_ate_entao')
)


x1 = x1.merge(
    reprov_disc_ate_entao,
    on=['id_discente', 'id_disciplina', 'ano_periodo'],
    how='left'
)





#### Agrupando po área do conhecimento

In [41]:
import pandas as pd

# ================================
# 1) ler o dicionário pronto
# ================================
dic_areas = pd.read_excel(r"C:\Users\Alef\Documents\Aprendizado de Máquina\dicionario_areas.xlsx",)

# manter só as colunas necessárias
dic_areas = dic_areas[["componente_curricular", "area_conhecimento"]].copy()

# ================================
# 2) ajustar nomes das colunas
# ================================
# No dicionário, a coluna está como:
# componente_curricular
#
# Na sua base, pelo que apareceu antes, pode estar como:
# nome_componete_curricular
#
# Se na sua base o nome for outro, ajuste abaixo.

x1 = x1.merge(
    dic_areas,
    left_on="nome_componete_curricular",
    right_on="componente_curricular",
    how="left"
)

# se não quiser manter a coluna duplicada do merge:
x1 = x1.drop(columns=["componente_curricular"])

# ================================
# 3) checar se alguma disciplina ficou sem área
# ================================
print("Quantidade sem área:", x1["area_conhecimento"].isna().sum())
print("\nDistribuição das áreas:")
print(x1["area_conhecimento"].value_counts(dropna=False))

# ================================
# 4) criar dummies da área
# ================================
x1 = pd.get_dummies(
    x1,
    columns=["area_conhecimento"],
    drop_first=True,
    dtype=int
)

print("\nBase após criação das dummies:")
print(x1.head())

Quantidade sem área: 225321

Distribuição das áreas:
area_conhecimento
NaN                         225321
saude                        94767
agro_ambiente                48576
humanas                      23472
exatas                       14737
sociais_aplicadas_gestao      9685
linguagens_artes              7850
tecnologia_engenharias        4743
educacao                      2482
formacao_geral                2300
Name: count, dtype: int64

Base após criação das dummies:
        id_discente discente_nivel  id_curso  id_curriculo  \
0  9519063299906826            NaN       NaN           NaN   
1  9519063299906826            NaN       NaN           NaN   
2  9519063299906826            NaN       NaN           NaN   
3  9519063299906826            NaN       NaN           NaN   
4  9519063299906826            NaN       NaN           NaN   

   id_estrutura_curricular  ano_ingresso  periodo_ingresso status_discente  \
0                      NaN           NaN               NaN            

#### Estimação com maior quantidade de variáveis, porém menos dados

In [ ]:
x1_2021 = x1[x1['ano_periodo']>=2021]

x1_2021=x1_2021[x1_2021['forma_ingresso'].notna()]



colunas_completas = [
    "id_discente",
    "discente_nivel",
    "id_curso",
    "id_estrutura_curricular",
    "ano_ingresso",
    "periodo_ingresso",
    "status_discente",
    "forma_ingresso",
    "media_geral",
    "ano_nascimento",
    "pais_origem_br",
    "id_disciplina",
    "ano",
    "periodo",
    "ch_aula",
    "ch_laboratorio",
    "ch_total",
    "cr_aula",
    "cr_laboratorio",
    "cr_estagio",
    "ch_ead",
    "nome_componete_curricular",
    "sexo_masculino",
    "estado_civil_outro",
    "raca_declarada_nan",
    "raca_declarada_nao_informado",
    "raca_declarada_negra",
    "raca_declarada_outra",
    "faixa_renda_familiar_2k_4k",
    "faixa_renda_familiar_4k_8k",
    "faixa_renda_familiar_8k_mais",
    "faixa_renda_familiar_ate_1k",
    "faixa_renda_familiar_nan",
    "id_curso_cursos_15467263.0",
    "id_curso_cursos_1958830.0",
    "id_curso_cursos_1958832.0",
    "id_curso_cursos_1996990.0",
    "id_curso_cursos_1997020.0",
    "id_curso_cursos_1997021.0",
    "id_curso_cursos_1997022.0",
    "id_curso_cursos_1997025.0",
    "id_curso_cursos_1997026.0",
    "id_curso_cursos_22279083.0",
    "id_curso_cursos_23657372.0",
    "id_curso_cursos_2558681.0",
    "id_curso_cursos_2663867.0",
    "id_curso_cursos_28685265.0",
    "id_curso_cursos_31102902.0",
    "id_curso_cursos_36687001.0",
    "id_curso_cursos_38121652.0",
    "id_curso_cursos_38121660.0",
    "id_curso_cursos_3926060.0",
    "id_curso_cursos_5050943.0",
    "id_curso_cursos_nan",
    "sigla_departamento_cpt-ets-coagcep",
    "sigla_departamento_nan",
    "situacao_rep_flt",
    "situacao_rep_md_flt",
    "situacao_reprovado",
    "reprovado",
    "tem_laboratorio",
    "tem_ead",
    "dummy_tecnico",
    "ano_periodo",
    "reprov_ate_entao",
    "reprov_disc_ate_entao",
    "area_conhecimento_educacao",
    "area_conhecimento_exatas",
    "area_conhecimento_formacao_geral",
    "area_conhecimento_humanas",
    "area_conhecimento_linguagens_artes",
    "area_conhecimento_saude",
    "area_conhecimento_sociais_aplicadas_gestao",
    "area_conhecimento_tecnologia_engenharias"
]

x1_2021=x1_2021[colunas_completas]


In [ ]:
x1_2021.forma_ingresso.value_coun

forma_ingresso
PROCESSO SELETIVO                                      108163
POR CONVÊNIO                                              671
PORTADOR DE DIPLOMA                                       135
TRANSFERÊNCIA                                              47
COOPERAÇÃO COM PAÍSES LUSÓFONOS E LATINO-AMERICANOS        45
REINGRESSO                                                 26
Name: count, dtype: int64

## Creditos

In [ ]:
# import pandas as pd
# import numpy as np

# # Garantir que as colunas estejam numéricas
# cols_credito = ['cr_aula', 'cr_laboratorio', 'cr_estagio']

# for col in cols_credito:
#     x1[col] = pd.to_numeric(x1[col], errors='coerce')

# # Criar a variável cr_total
# x1['cr_total'] = x1[['cr_aula', 'cr_laboratorio', 'cr_estagio']].fillna(0).sum(axis=1)

# # Opcional: se quiser transformar 0 em NaN quando os 3 créditos estiverem ausentes
# mask_todos_nan = x1[cols_credito].isna().all(axis=1)
# x1.loc[mask_todos_nan, 'cr_total'] = np.nan

# # Conferir
# print(x1[['cr_aula', 'cr_laboratorio', 'cr_estagio', 'cr_total']].value_counts())

cr_aula  cr_laboratorio  cr_estagio  cr_total
0.0      0.0             0.0         0.0         118555
2.0      0.0             0.0         2.0           4930
4.0      0.0             0.0         4.0           2093
3.0      0.0             0.0         3.0           2076
1.0      0.0             0.0         1.0           1423
6.0      0.0             0.0         6.0            963
5.0      0.0             0.0         5.0            695
8.0      0.0             0.0         8.0            554
0.0      1.0             0.0         1.0            357
7.0      0.0             0.0         7.0            165
13.0     0.0             0.0         13.0           153
0.0      2.0             0.0         2.0            143
Name: count, dtype: int64


## Preparando base para estimação

In [28]:


# variáveis que podem entrar no modelo
colunas_modelo = [
    'reprovado',
    "sexo_masculino",
    "estado_civil_outro",
    "raca_declarada_nan",
    "raca_declarada_nao_informado",
    "raca_declarada_negra",
    "raca_declarada_outra",
    "dummy_tecnico",
    "tem_laboratorio",
    "tem_ead",
    "faixa_renda_familiar_2k_4k",
    "faixa_renda_familiar_4k_8k",
    "faixa_renda_familiar_8k_mais",
    "faixa_renda_familiar_ate_1k",
    "faixa_renda_familiar_nan",
    "ano_periodo",
    "reprov_ate_entao",
    "reprov_disc_ate_entao",
    "id_curso_cursos_15467263.0",
    "id_curso_cursos_1958830.0",
    "id_curso_cursos_1958832.0",
    "id_curso_cursos_1996990.0",
    "id_curso_cursos_1997020.0",
    "id_curso_cursos_1997021.0",
    "id_curso_cursos_1997022.0",
    "id_curso_cursos_1997025.0",
    "id_curso_cursos_1997026.0",
    "id_curso_cursos_22279083.0",
    "id_curso_cursos_23657372.0",
    "id_curso_cursos_2558681.0",
    "id_curso_cursos_2663867.0",
    "id_curso_cursos_28685265.0",
    "id_curso_cursos_31102902.0",
    "id_curso_cursos_36687001.0",
    "id_curso_cursos_38121652.0",
    "id_curso_cursos_38121660.0",
    "id_curso_cursos_3926060.0",
    "id_curso_cursos_5050943.0",
    "id_curso_cursos_nan",
    'area_conhecimento_educacao',
    'area_conhecimento_exatas', 'area_conhecimento_formacao_geral',
    'area_conhecimento_humanas', 'area_conhecimento_linguagens_artes',
    'area_conhecimento_saude', 'area_conhecimento_sociais_aplicadas_gestao',
    'area_conhecimento_tecnologia_engenharias',
    'area_conhecimento_educacao', 'area_conhecimento_exatas',
    'area_conhecimento_formacao_geral', 'area_conhecimento_humanas',
    'area_conhecimento_linguagens_artes', 'area_conhecimento_saude',
    'area_conhecimento_sociais_aplicadas_gestao',
    'area_conhecimento_tecnologia_engenharias'
]


# matriz de regressoras
X = x1[colunas_modelo].copy()


### Estimando para depois de 2020

In [12]:
X=X.loc[X['ano_periodo']>=2021].copy()

In [17]:
X.columns

Index(['reprovado', 'sexo_masculino', 'estado_civil_outro',
       'raca_declarada_nan', 'raca_declarada_nao_informado',
       'raca_declarada_negra', 'raca_declarada_outra', 'dummy_tecnico',
       'tem_laboratorio', 'tem_ead', 'faixa_renda_familiar_2k_4k',
       'faixa_renda_familiar_4k_8k', 'faixa_renda_familiar_8k_mais',
       'faixa_renda_familiar_ate_1k', 'faixa_renda_familiar_nan',
       'ano_periodo', 'reprov_ate_entao', 'reprov_disc_ate_entao',
       'id_curso_cursos_15467263.0', 'id_curso_cursos_1958830.0',
       'id_curso_cursos_1958832.0', 'id_curso_cursos_1996990.0',
       'id_curso_cursos_1997020.0', 'id_curso_cursos_1997021.0',
       'id_curso_cursos_1997022.0', 'id_curso_cursos_1997025.0',
       'id_curso_cursos_1997026.0', 'id_curso_cursos_22279083.0',
       'id_curso_cursos_23657372.0', 'id_curso_cursos_2558681.0',
       'id_curso_cursos_2663867.0', 'id_curso_cursos_28685265.0',
       'id_curso_cursos_31102902.0', 'id_curso_cursos_36687001.0',
       'id_c

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1) preparar X e y
# --------------------------------------------------
# Supondo que X já contém a variável alvo 'reprovado'
# Se quiser, pode fazer uma cópia:
X = X.copy()

# separar preditores e alvo
X_model = X.drop(columns=['reprovado']).copy()
y = X['reprovado'].copy()

# --------------------------------------------------
# 2) divisão aleatória treino-teste
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.30,      # 30% teste, 70% treino
    random_state=1,
    stratify=y           # mantém a proporção de reprovados
)

# checagens
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nTaxa de reprovação no treino:", y_train.mean())
print("Taxa de reprovação no teste :", y_test.mean())

# --------------------------------------------------
# 3) validação cruzada
# --------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# --------------------------------------------------
# 4) pipeline
# --------------------------------------------------
pipe_logit = Pipeline([
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=1,
        solver="liblinear"
    ))
])

# --------------------------------------------------
# 5) grade de hiperparâmetros
# --------------------------------------------------
param_grid = {
    "model__C": [0.01, 0.5, 1],
    "model__penalty": ["l2"]
}

# --------------------------------------------------
# 6) grid search
# --------------------------------------------------
grid_logit = GridSearchCV(
    estimator=pipe_logit,
    param_grid=param_grid,
    cv=cv,
    scoring="recall",   # aqui você está otimizando recall
    n_jobs=-1,
    verbose=1
)

# --------------------------------------------------
# 7) ajuste do modelo
# --------------------------------------------------
grid_logit.fit(X_train, y_train)

# melhores resultados
print("\nMelhores parâmetros:", grid_logit.best_params_)
print("Melhor recall médio na validação cruzada:", grid_logit.best_score_)

# --------------------------------------------------
# 8) previsões no teste
# --------------------------------------------------
y_pred = grid_logit.predict(X_test)
y_prob = grid_logit.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# 9) métricas
# --------------------------------------------------
print("\nMétricas no teste:")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, zero_division=0))

# --------------------------------------------------
# 10) coeficientes da melhor regressão logística
# --------------------------------------------------
best_logit = grid_logit.best_estimator_.named_steps["model"]

coeficientes = pd.DataFrame({
    "variavel": X_train.columns,
    "coeficiente": best_logit.coef_[0],
    "odds_ratio": np.exp(best_logit.coef_[0])
}).sort_values("coeficiente", ascending=False)

print("\nCoeficientes da regressão logística:")
print(coeficientes)

# --------------------------------------------------
# 11) probabilidades previstas
# --------------------------------------------------
resultado_teste = X_test.copy()
resultado_teste["y_real"] = y_test.values
resultado_teste["y_pred"] = y_pred
resultado_teste["prob_reprovado"] = y_prob

print("\nAmostra dos resultados:")
print(resultado_teste.head())

y_test_2021 = y_test.copy()
y_pred_logit_2021 = y_pred.copy()
y_prob_logit_2021 = y_prob.copy()

X_train: (191627, 54)
y_train: (191627,)
X_test : (82127, 54)
y_test : (82127,)

Taxa de reprovação no treino: 0.10858073236026239
Taxa de reprovação no teste : 0.10858791871126426
Fitting 5 folds for each of 3 candidates, totalling 15 fits


c:\Users\Alef\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



Melhores parâmetros: {'model__C': 1, 'model__penalty': 'l2'}
Melhor recall médio na validação cruzada: 0.4562061491103166

Métricas no teste:
Accuracy : 0.7973260925152508
Precision: 0.3241545673842793
Recall   : 0.7986095537115945
F1-score : 0.4611350318883745
ROC-AUC  : 0.8697672329713642

Matriz de confusão:
[[58360 14849]
 [ 1796  7122]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.97      0.80      0.88     73209
           1       0.32      0.80      0.46      8918

    accuracy                           0.80     82127
   macro avg       0.65      0.80      0.67     82127
weighted avg       0.90      0.80      0.83     82127


Coeficientes da regressão logística:
                                      variavel  coeficiente  odds_ratio
34                  id_curso_cursos_38121660.0     2.350985   10.495902
13                    faixa_renda_familiar_nan     1.582827    4.868701
33                  id_curso_cursos_38121652.

In [16]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1) preparar X e y
# --------------------------------------------------
# Supondo que X contém a variável alvo 'reprovado'
X = X.copy()

X_model = X.drop(columns=['reprovado']).copy()
y = X['reprovado'].copy()

# ----------------------------------------------------------
# 2) divisão aleatória treino-teste                        |       
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.30,
    random_state=1,
    stratify=y
)

# checagens
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nTaxa de reprovação no treino:", y_train.mean())
print("Taxa de reprovação no teste :", y_test.mean())

# --------------------------------------------------
# 3) validação cruzada
# --------------------------------------------------
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)

# --------------------------------------------------
# 4) pipeline Random Forest
# --------------------------------------------------
pipe_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        random_state=1,
        class_weight="balanced",
        n_jobs=1
    ))
])

# --------------------------------------------------
# 5) grade de hiperparâmetros
# --------------------------------------------------
param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [4, 8, None],
    "model__min_samples_leaf": [1, 5, 8]
}

# --------------------------------------------------
# 6) grid search
# --------------------------------------------------
grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid,
    cv=cv,
    scoring="recall",
    n_jobs=-1,
    verbose=1
)

# --------------------------------------------------
# 7) ajuste do modelo
# --------------------------------------------------
grid_rf.fit(X_train, y_train)

# --------------------------------------------------
# 8) melhores resultados
# --------------------------------------------------
print("\nMelhores parâmetros modelo RF:", grid_rf.best_params_)
print("Melhor recall médio na validação cruzada:", grid_rf.best_score_)

# --------------------------------------------------
# 9) previsões no teste
# --------------------------------------------------
y_pred = grid_rf.predict(X_test)
y_prob = grid_rf.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# 10) métricas
# --------------------------------------------------
print("\nMétricas no teste")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, zero_division=0))


y_test_2021 = y_test.copy()
y_pred_rf_2021 = y_pred.copy()
y_prob_rf_2021 = y_prob.copy()

X_train: (191627, 54)
y_train: (191627,)
X_test : (82127, 54)
y_test : (82127,)

Taxa de reprovação no treino: 0.10858073236026239
Taxa de reprovação no teste : 0.10858791871126426
Fitting 3 folds for each of 18 candidates, totalling 54 fits

Melhores parâmetros modelo RF: {'model__max_depth': 8, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
Melhor recall médio na validação cruzada: 0.8973423371355979

Métricas no teste
Accuracy : 0.7387582646389129
Precision: 0.27976671468221903
Recall   : 0.8929132092397398
F1-score : 0.4260453171397234
ROC-AUC  : 0.8865288029937827

Matriz de confusão:
[[52709 20500]
 [  955  7963]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.98      0.72      0.83     73209
           1       0.28      0.89      0.43      8918

    accuracy                           0.74     82127
   macro avg       0.63      0.81      0.63     82127
weighted avg       0.91      0.74      0.79     82127



## XGBOOST

In [29]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1) preparar X e y
# --------------------------------------------------
# Supondo que X contém a variável alvo 'reprovado'
X = X.copy()

X_model = X.drop(columns=['reprovado']).copy()
y = X['reprovado'].copy()

# --------------------------------------------------
# 2) divisão aleatória treino-teste
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.30,
    random_state=1,
    stratify=y
)

# checagens
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nTaxa de reprovação no treino:", y_train.mean())
print("Taxa de reprovação no teste :", y_test.mean())

# --------------------------------------------------
# 3) calcular scale_pos_weight
# --------------------------------------------------
# útil para lidar com desbalanceamento no XGBoost
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print("\nscale_pos_weight:", scale_pos_weight)

# --------------------------------------------------
# 4) validação cruzada
# --------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# --------------------------------------------------
# 5) pipeline XGBoost
# --------------------------------------------------
pipe_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        random_state=1,
        n_jobs=-1,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    ))
])

# --------------------------------------------------
# 6) grade de hiperparâmetros
# --------------------------------------------------
param_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [4, 8],
    "model__learning_rate": [0.05, 0.1],
    "model__min_child_weight": [1, 5],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

# --------------------------------------------------
# 7) grid search
# --------------------------------------------------
grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid,
    cv=cv,
    scoring="recall",
    n_jobs=-1,
    verbose=1
)

# --------------------------------------------------
# 8) ajuste do modelo
# --------------------------------------------------
grid_xgb.fit(X_train, y_train)

# --------------------------------------------------
# 9) melhores resultados
# --------------------------------------------------
print("\nMelhores parâmetros modelo XGBoost:", grid_xgb.best_params_)
print("Melhor recall médio na validação cruzada:", grid_xgb.best_score_)

# --------------------------------------------------
# 10) previsões no teste
# --------------------------------------------------
y_pred = grid_xgb.predict(X_test)
y_prob = grid_xgb.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# 11) métricas
# --------------------------------------------------
print("\nMétricas no teste")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, zero_division=0))


y_test_2021 = y_test.copy()
y_pred_xgb_2021 = y_pred.copy()
y_prob_xgb_2021 = y_prob.copy()

X_train: (303753, 54)
y_train: (303753,)
X_test : (130180, 54)
y_test : (130180,)

Taxa de reprovação no treino: 0.12890407666755554
Taxa de reprovação no teste : 0.12890612997388232

scale_pos_weight: 6.757706550887498
Fitting 5 folds for each of 64 candidates, totalling 320 fits

Melhores parâmetros modelo XGBoost: {'model__colsample_bytree': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 8, 'model__min_child_weight': 5, 'model__n_estimators': 500, 'model__subsample': 0.8}
Melhor recall médio na validação cruzada: 0.842140211978036

Métricas no teste
Accuracy : 0.8221693040405592
Precision: 0.40884499785315587
Recall   : 0.8511411715630773
F1-score : 0.5523629050970686
ROC-AUC  : 0.908117096408056

Matriz de confusão:
[[92747 20652]
 [ 2498 14283]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.97      0.82      0.89    113399
           1       0.41      0.85      0.55     16781

    accuracy                           

## Estimando Modelos para todo o período dividindo a base entre antes de 2020 e após

In [29]:
# matriz de regressoras
X = x1[colunas_modelo].copy()

#### Logit

In [30]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import pandas as pd
import numpy as np

# extrair apenas o ano
x1["ano_ref"] = np.floor(x1["ano_periodo"]).astype(int)

# dividir temporalmente
mask_train = x1["ano_ref"] < 2020
mask_test  = x1["ano_ref"] >= 2020


X_train = X.loc[mask_train].copy()
X_train = X_train.drop(columns=['reprovado'],axis=1)
y_train = X.loc[mask_train].copy()['reprovado']

X_test = X.loc[mask_test].copy()
X_test = X_test.drop(columns=['reprovado'],axis=1)
y_test = X.loc[mask_test].copy()['reprovado']

# checagens
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nTaxa de reprovação no treino:", y_train.mean())
print("Taxa de reprovação no teste :", y_test.mean())

# validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# pipeline
pipe_logit = Pipeline([
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=1,
        solver="liblinear"
    ))
])

# grade de hiperparâmetros
param_grid = {
    "model__C": [0.01,0.5,1],
    "model__penalty": ["l2"]
}

# grid search
grid_logit = GridSearchCV(
    estimator=pipe_logit,
    param_grid=param_grid,
    cv=cv,
    scoring="recall",
    n_jobs=-1,
    verbose=1
)

# ajuste do modelo
grid_logit.fit(X_train, y_train)

# melhores resultados
print("\nMelhores parâmetros:", grid_logit.best_params_)
print("Melhor F1 médio na validação cruzada:", grid_logit.best_score_)

# previsões no teste
y_pred = grid_logit.predict(X_test)
y_prob = grid_logit.predict_proba(X_test)[:, 1]

# métricas
print("\nMétricas no teste:")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, zero_division=0))

# coeficientes da melhor regressão logística
best_logit = grid_logit.best_estimator_.named_steps["model"]

coeficientes = pd.DataFrame({
    "variavel": X_train.columns,
    "coeficiente": best_logit.coef_[0],
    "odds_ratio": np.exp(best_logit.coef_[0])
}).sort_values("coeficiente", ascending=False)

print("\nCoeficientes da regressão logística:")
print(coeficientes)

# probabilidades previstas
resultado_teste = X_test.copy()
resultado_teste["y_real"] = y_test.values
resultado_teste["y_pred"] = y_pred
resultado_teste["prob_reprovado"] = y_prob

print("\nAmostra dos resultados:")
print(resultado_teste.head())


y_test_all = y_test.copy()
y_pred_logit_all = y_pred.copy()
y_prob_logit_all = y_prob.copy()

X_train: (82091, 38)
y_train: (82091,)
X_test : (275337, 38)
y_test : (275337,)

Taxa de reprovação no treino: 0.19101972201581172
Taxa de reprovação no teste : 0.08313811801537752
Fitting 5 folds for each of 3 candidates, totalling 15 fits


c:\Users\Alef\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



Melhores parâmetros: {'model__C': 1, 'model__penalty': 'l2'}
Melhor F1 médio na validação cruzada: 0.6981060686148861

Métricas no teste:
Accuracy : 0.868924263720459
Precision: 0.328588867821615
Recall   : 0.5526626184963522
F1-score : 0.4121383893666927
ROC-AUC  : 0.7430896433440084

Matriz de confusão:
[[226596  25850]
 [ 10240  12651]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.96      0.90      0.93    252446
           1       0.33      0.55      0.41     22891

    accuracy                           0.87    275337
   macro avg       0.64      0.73      0.67    275337
weighted avg       0.90      0.87      0.88    275337


Coeficientes da regressão logística:
                        variavel  coeficiente  odds_ratio
37           id_curso_cursos_nan     1.814432    6.137592
13      faixa_renda_familiar_nan     1.706486    5.509564
11  faixa_renda_familiar_8k_mais     0.860604    2.364587
20     id_curso_cursos_1996990.

#### random forest

In [31]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1) preparar corte temporal
# --------------------------------------------------
x1["ano_ref"] = np.floor(x1["ano_periodo"]).astype(int)

mask_train = x1["ano_ref"] < 2020
mask_test  = x1["ano_ref"] >= 2020

X_train = X.loc[mask_train].copy()
X_train = X_train.drop(columns=['reprovado'], axis=1)
y_train = X.loc[mask_train, 'reprovado'].copy()

X_test = X.loc[mask_test].copy()
X_test = X_test.drop(columns=['reprovado'], axis=1)
y_test = X.loc[mask_test, 'reprovado'].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nTaxa de reprovação no treino:", y_train.mean())
print("Taxa de reprovação no teste :", y_test.mean())

# --------------------------------------------------
# 2) validação cruzada
# --------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# --------------------------------------------------
# 3) pipeline Random Forest
# --------------------------------------------------
pipe_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        random_state=1,
        class_weight="balanced",
        n_jobs=-1
    ))
])

# --------------------------------------------------
# 4) grade de hiperparâmetros
# --------------------------------------------------
param_grid_rf = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [8, 12, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"]
}

# --------------------------------------------------
# 5) grid search
# --------------------------------------------------
grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring="recall",
    n_jobs=-1,
    verbose=1
)

# ajuste
grid_rf.fit(X_train, y_train)

print("\nMelhores parâmetros RF:", grid_rf.best_params_)
print("Melhor recall médio na validação cruzada:", grid_rf.best_score_)

# --------------------------------------------------
# 6) previsões no teste
# --------------------------------------------------
y_pred_rf = grid_rf.predict(X_test)
y_prob_rf = grid_rf.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# 7) métricas
# --------------------------------------------------
print("\nMétricas no teste - Random Forest")
print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred_rf, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_rf))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

# --------------------------------------------------
# 8) importância das variáveis
# --------------------------------------------------
best_rf = grid_rf.best_estimator_.named_steps["model"]

importancias_rf = pd.DataFrame({
    "variavel": X_train.columns,
    "importancia": best_rf.feature_importances_
}).sort_values("importancia", ascending=False)

print("\nImportância das variáveis - Random Forest:")
print(importancias_rf.head(30))

# --------------------------------------------------
# 9) resultados no teste
# --------------------------------------------------
resultado_rf = X_test.copy()
resultado_rf["y_real"] = y_test.values
resultado_rf["y_pred"] = y_pred_rf
resultado_rf["prob_reprovado"] = y_prob_rf

print("\nAmostra dos resultados:")
print(resultado_rf.head())


y_test_all = y_test.copy()
y_pred_rf_all = y_pred_rf.copy()
y_prob_rf_all = y_prob_rf.copy()

X_train: (82091, 38)
y_train: (82091,)
X_test : (275337, 38)
y_test : (275337,)

Taxa de reprovação no treino: 0.19101972201581172
Taxa de reprovação no teste : 0.08313811801537752
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Melhores parâmetros RF: {'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Melhor recall médio na validação cruzada: 0.7610487767788021

Métricas no teste - Random Forest
Accuracy : 0.8882169850038316
Precision: 0.37504356918787035
Recall   : 0.5170591061989428
F1-score : 0.43474747474747477
ROC-AUC  : 0.7868088687123678

Matriz de confusão:
[[232723  19723]
 [ 11055  11836]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.95      0.92      0.94    252446
           1       0.38      0.52      0.43     22891

    accuracy                           0.89    275337
   macro avg       0.66      0.72    

In [19]:
print("Melhores parâmetros modelo RF:", grid_rf.best_params_)
print("Melhor Scores de Performance:", grid_rf.best_score_)

Melhores parâmetros modelo RF: {'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Melhor Scores de Performance: 0.7610487767788021


## Estimando modelo XGBOOST

In [32]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1) preparar corte temporal
# --------------------------------------------------
x1["ano_ref"] = np.floor(x1["ano_periodo"]).astype(int)

mask_train = x1["ano_ref"] < 2020
mask_test  = x1["ano_ref"] >= 2020

X_train = X.loc[mask_train].copy()
X_train = X_train.drop(columns=['reprovado'], axis=1)
y_train = X.loc[mask_train, 'reprovado'].copy()

X_test = X.loc[mask_test].copy()
X_test = X_test.drop(columns=['reprovado'], axis=1)
y_test = X.loc[mask_test, 'reprovado'].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nTaxa de reprovação no treino:", y_train.mean())
print("Taxa de reprovação no teste :", y_test.mean())

# --------------------------------------------------
# 2) peso da classe positiva
# --------------------------------------------------
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("\nscale_pos_weight:", scale_pos_weight)

# --------------------------------------------------
# 3) validação cruzada
# --------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# --------------------------------------------------
# 4) pipeline XGBoost
# --------------------------------------------------
pipe_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=1,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist"
    ))
])

# --------------------------------------------------
# 5) grade de hiperparâmetros
# --------------------------------------------------
param_grid_xgb = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

# --------------------------------------------------
# 6) grid search
# --------------------------------------------------
grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    cv=cv,
    scoring="recall",
    n_jobs=-1,
    verbose=1
)

# ajuste
grid_xgb.fit(X_train, y_train)

print("\nMelhores parâmetros XGBoost:", grid_xgb.best_params_)
print("Melhor recall médio na validação cruzada:", grid_xgb.best_score_)

# --------------------------------------------------
# 7) previsões no teste
# --------------------------------------------------
y_pred_xgb = grid_xgb.predict(X_test)
y_prob_xgb = grid_xgb.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# 8) métricas
# --------------------------------------------------
print("\nMétricas no teste - XGBoost")
print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_xgb, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred_xgb, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_xgb))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_xgb, zero_division=0))

# --------------------------------------------------
# 9) importância das variáveis
# --------------------------------------------------
best_xgb = grid_xgb.best_estimator_.named_steps["model"]

importancias_xgb = pd.DataFrame({
    "variavel": X_train.columns,
    "importancia": best_xgb.feature_importances_
}).sort_values("importancia", ascending=False)

print("\nImportância das variáveis - XGBoost:")
print(importancias_xgb.head(30))

# --------------------------------------------------
# 10) resultados no teste
# --------------------------------------------------
resultado_xgb = X_test.copy()
resultado_xgb["y_real"] = y_test.values
resultado_xgb["y_pred"] = y_pred_xgb
resultado_xgb["prob_reprovado"] = y_prob_xgb

print("\nAmostra dos resultados:")
print(resultado_xgb.head())

y_test_all = y_test.copy()
y_pred_xgb_all = y_pred_xgb.copy()
y_prob_xgb_all = y_prob_xgb.copy()

X_train: (82091, 38)
y_train: (82091,)
X_test : (275337, 38)
y_test : (275337,)

Taxa de reprovação no treino: 0.19101972201581172
Taxa de reprovação no teste : 0.08313811801537752

scale_pos_weight: 4.235061539442637
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Melhores parâmetros XGBoost: {'model__colsample_bytree': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 7, 'model__n_estimators': 300, 'model__subsample': 0.8}
Melhor recall médio na validação cruzada: 0.7357955451067899

Métricas no teste - XGBoost
Accuracy : 0.8996175595724513
Precision: 0.36806713348894077
Recall   : 0.28932768336900966
F1-score : 0.3239819004524887
ROC-AUC  : 0.7780060381992666

Matriz de confusão:
[[241075  11371]
 [ 16268   6623]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95    252446
           1       0.37      0.29      0.32     22891

    accuracy                           0.90    275337
   macr

#### Comparando resultados

In [33]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ---------------------------------------------------
# 1) função para calcular métricas
# ---------------------------------------------------
def avaliar_modelo(nome_modelo, tipo_base, y_true, y_pred, y_prob):
    return {
        "tipo_base": tipo_base,
        "modelo": nome_modelo,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

# ---------------------------------------------------
# 2) lista para armazenar resultados
# ---------------------------------------------------
resultados = []

# ===================================================
# A) MODELOS ESTIMADOS COM TODO O PERÍODO
# ===================================================
# Substitua os objetos abaixo pelos nomes que você usou no seu código

# LOGIT - todo período
resultados.append(
    avaliar_modelo(
        nome_modelo="Logit",
        tipo_base="Todo o período",
        y_true=y_test_all,
        y_pred=y_pred_logit_all,
        y_prob=y_prob_logit_all
    )
)

# RANDOM FOREST - todo período
resultados.append(
    avaliar_modelo(
        nome_modelo="Random Forest",
        tipo_base="Todo o período",
        y_true=y_test_all,
        y_pred=y_pred_rf_all,
        y_prob=y_prob_rf_all
    )
)

# XGBOOST - todo período
resultados.append(
    avaliar_modelo(
        nome_modelo="XGBoost",
        tipo_base="Todo o período",
        y_true=y_test_all,
        y_pred=y_pred_xgb_all,
        y_prob=y_prob_xgb_all
    )
)

# ===================================================
# B) MODELOS ESTIMADOS APENAS COM DADOS >= 2021
# ===================================================

# LOGIT - 2021+
resultados.append(
    avaliar_modelo(
        nome_modelo="Logit",
        tipo_base="2021 em diante",
        y_true=y_test_2021,
        y_pred=y_pred_logit_2021,
        y_prob=y_prob_logit_2021
    )
)

# RANDOM FOREST - 2021+
resultados.append(
    avaliar_modelo(
        nome_modelo="Random Forest",
        tipo_base="2021 em diante",
        y_true=y_test_2021,
        y_pred=y_pred_rf_2021,
        y_prob=y_prob_rf_2021
    )
)

# XGBOOST - 2021+
resultados.append(
    avaliar_modelo(
        nome_modelo="XGBoost",
        tipo_base="2021 em diante",
        y_true=y_test_2021,
        y_pred=y_pred_xgb_2021,
        y_prob=y_prob_xgb_2021
    )
)

# ---------------------------------------------------
# 3) transformar em DataFrame
# ---------------------------------------------------
comparacao_modelos = pd.DataFrame(resultados)

# arredondar
comparacao_modelos = comparacao_modelos.round(4)

# ordenar
comparacao_modelos = comparacao_modelos.sort_values(
    by=["tipo_base", "f1_score", "recall", "roc_auc"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print(comparacao_modelos)

        tipo_base         modelo  accuracy  precision  recall  f1_score  \
0  2021 em diante        XGBoost    0.8260     0.3040  0.8546    0.4484   
1  2021 em diante          Logit    0.8283     0.2981  0.7932    0.4333   
2  2021 em diante  Random Forest    0.7922     0.2684  0.8749    0.4107   
3  Todo o período  Random Forest    0.8882     0.3750  0.5171    0.4347   
4  Todo o período          Logit    0.8689     0.3286  0.5527    0.4121   
5  Todo o período        XGBoost    0.8996     0.3681  0.2893    0.3240   

   roc_auc  
0   0.8936  
1   0.8348  
2   0.8917  
3   0.7868  
4   0.7431  
5   0.7780  
